## Browser Profile을 사용하는 AgentCore Browser Tool

이 예제에서는 AgentCore Browser에서 browser profile을 사용하는 방법을 알아봅니다.
이 기능을 사용하면 여러 브라우저 세션에서 browser profile 데이터를 유지하고 재사용할 수 있습니다. Browser profile에는 cookie 및 local storage를 비롯한 세션 정보가 저장됩니다.

**시작하기 전에 CloudFormation stack을 실행해 이 튜토리얼에서 사용할 간단한 가상 e-commerce를 배포해야 합니다.**

CloudFormation output에서 CloudFront distribution 이름을 가져옵니다.

![cfn_outputs](img/cfn_outputs.png)

또는 [deploy.sh](sample-ecommerce/deploy.sh) 스크립트 output에서 CloudFront distribution 이름을 가져와 다음 셀에 입력합니다.

In [ ]:
CFN_URL = "<your-cloud-front-url>"

시작하려면 dependency를 설치하고 **kernel을 다시 시작하세요**.

In [ ]:
!pip install -qU -r requirements.txt

### 1. 사용자 지정 브라우저 생성

이 단계에서는 Notebook 전체에서 사용할 전역 변수를 선언합니다.

In [ ]:
import boto3
import json
import sys
from botocore.exceptions import ClientError


sys.path.append("../helpers/")

iam_boto3 = boto3.client("iam")
s3 = boto3.client("s3")
browser_boto3 = boto3.client("bedrock-agentcore-control")
browser_cli = boto3.client("bedrock-agentcore")

session = boto3.Session()
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
REGION = session.region_name

BROWSER_NAME = "browser_with_profiles"
BROWSER_PROFILE_NAME = "profile_sample"
BUCKET_NAME = f"ac-browser-demos-{ACCOUNT_ID}-{REGION}"
AC_ROLE_NAME = "ac-browser-execution-role"

#### 1.1 S3 Bucket 생성

나중에 다운로드할 브라우저 녹화를 저장할 S3 Bucket이 없다면 새로 생성해야 합니다.

In [ ]:
try:
    # Bucket이 있는지 확인
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket {BUCKET_NAME} already exists")
except ClientError:
    # Bucket 생성
    create_params = {"Bucket": BUCKET_NAME}
    if REGION != "us-east-1":
        create_params["CreateBucketConfiguration"] = {"LocationConstraint": REGION}
    s3.create_bucket(**create_params)
    print(f"Bucket {BUCKET_NAME} created in {REGION}")

#### 1.2 IAM role 생성

그런 다음 AgentCore Browser에 연결할 사용자 지정 IAM role을 생성합니다.

In [ ]:
try:
    # Trust policy 정의
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }

    # Role 생성
    browser_role = iam_boto3.create_role(RoleName=AC_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy))

    browser_role_arn = browser_role["Role"]["Arn"]

    print(f"Role ARN: {browser_role_arn}")

    # 녹화를 위한 S3 policy
    ac_browser_policies = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "s3:PutObject",
                    "s3:GetObject",
                    "s3:ListBucket",
                    "s3:ListMultipartUploadParts",
                    "s3:AbortMultipartUpload",
                ],
                "Resource": [
                    f"arn:aws:s3:::{BUCKET_NAME}",
                    f"arn:aws:s3:::{BUCKET_NAME}/*",
                ],
            },
            {
                "Sid": "BedrockAgentCoreBrowserProfileUsageAccess",
                "Effect": "Allow",
                "Action": [
                    "bedrock-agentcore:StartBrowserSession",
                    "bedrock-agentcore:SaveBrowserSessionProfile",
                ],
                "Resource": [
                    f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:browser-profile/{BROWSER_PROFILE_NAME}",
                    f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:browser-custom/{BROWSER_NAME}",
                ],
            },
        ],
    }

    # S3 inline policy 추가
    iam_boto3.put_role_policy(
        RoleName=AC_ROLE_NAME,
        PolicyName="ac_custom_policies",
        PolicyDocument=json.dumps(ac_browser_policies),
    )

    # Bedrock managed policy 연결
    iam_boto3.attach_role_policy(
        RoleName=AC_ROLE_NAME,
        PolicyArn="arn:aws:iam::aws:policy/AmazonBedrockFullAccess",
    )

except ClientError as e:
    print(f"Exception: {e}")
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        browser_role_arn = iam_boto3.get_role(RoleName=AC_ROLE_NAME)["Role"]["Arn"]
        print(f"Arn captured: {browser_role_arn}")

IAM role이 전파되도록 10초 동안 기다립니다.

In [ ]:
import time

time.sleep(10)

#### 1.3 사용자 지정 AgentCore Browser 생성

이 예제에서는 사용자 지정 브라우저를 생성하지만 Browser Profile 기능은 managed browser(`aws.browser.v1`)에서도 작동합니다.

In [ ]:
created_browser = browser_boto3.create_browser(
    name=BROWSER_NAME,
    executionRoleArn=browser_role_arn,
    networkConfiguration={"networkMode": "PUBLIC"},
    recording={
        "enabled": True,
        "s3Location": {"bucket": BUCKET_NAME, "prefix": "browser_recordings/"},
    },
)

browser_id = created_browser["browserId"]
print(f"Browser ID: {browser_id}")

#### 1.4 Browser profile 생성

In [ ]:
created_profile = browser_boto3.create_browser_profile(name=BROWSER_PROFILE_NAME, description="Example profile")

profile_id = created_profile["profileId"]
print(f"Created profile: {profile_id}")

### 2. 테스트

테스트를 시작하기 위해 새 브라우저 세션을 시작합니다.

In [ ]:
response = browser_cli.start_browser_session(browserIdentifier=browser_id)

session_id = response["sessionId"]
print(f"Session ID: {session_id}")

다음 셀에서는 IAM 자격 증명을 추가하기 위해 SigV4로 request에 서명합니다.

In [ ]:
import browser_helper as helper

url = helper.get_url(browser_id, session_id)
headers = helper.get_signed_headers(url)
headers

#### 2.1 단일 세션에서 테스트

이제 Playwright를 사용해 sample e-commerce 탐색을 시뮬레이션합니다.
[Playwright](https://playwright.dev/docs/intro)는 AgentCore Browser에서 지원하는 Web Testing and Automation framework입니다.
다음 순서로 진행합니다.
1. *Cart* 페이지로 이동해 장바구니가 비어 있는지 확인합니다.
1. 장바구니에 Echo Dot을 추가합니다.
1. 장바구니를 다시 확인해 제품이 추가되었는지 살펴봅니다.

In [ ]:
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(url, headers=headers)
    page = browser.contexts[0].pages[0] if browser.contexts else await browser.new_context().new_page()

    try:
        # 1. Home page로 이동
        await page.goto(f"{CFN_URL}/#home", wait_until="domcontentloaded")

        await page.wait_for_timeout(2000)

        # 2. 첫 번째 item 추가
        button = page.locator('button[onclick="addToCart(2)"]')
        await button.wait_for(state="visible")
        await button.click()

        await page.wait_for_timeout(2000)

        # 3. 두 번째 item 추가
        button = page.locator('button[onclick="addToCart(4)"]')
        await button.wait_for(state="visible")
        await button.click()

        # 4. Cart 확인
        view_cart_button = page.locator("#viewCart")
        await view_cart_button.wait_for(state="visible")
        await view_cart_button.click()
        await page.wait_for_timeout(2000)

        # 5. Home page로 돌아가기
        view_cart_button = page.locator("#backToProducts")
        await view_cart_button.wait_for(state="visible")
        await view_cart_button.click()

        # 6. Cart 상태가 로컬에 저장되는지 확인
        await page.evaluate("localStorage.setItem('cart', JSON.stringify(cart))")
        await page.wait_for_timeout(500)

    except Exception as error:
        print(f"Error during navigation: {error}")
        raise

#### 2.2 Profile에 세션 저장

이제 이 세션을 앞에서 생성한 profile에 저장합니다.

In [ ]:
response = browser_cli.save_browser_session_profile(
    profileIdentifier=profile_id, browserIdentifier=browser_id, sessionId=session_id
)

print("Profile saved successfully")

#### 2.3 세션 중지

세션을 중지합니다.

In [ ]:
stoped_session = browser_cli.stop_browser_session(browserIdentifier=browser_id, sessionId=session_id)
stoped_session

#### 2.4 새 세션 시작

새 세션을 시작하고 장바구니가 저장된 browser profile을 추가합니다.

In [ ]:
response = browser_cli.start_browser_session(
    browserIdentifier=browser_id, profileConfiguration={"profileIdentifier": profile_id}
)

session_id = response["sessionId"]
print(f"Session ID: {session_id}")

In [ ]:
import browser_helper as helper

url = helper.get_url(browser_id, session_id)
headers = helper.get_signed_headers(url)
headers

#### 2.5 장바구니 확인

제품이 장바구니에 이미 선택되어 있는지 확인합니다.

In [ ]:
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(url, headers=headers)
    page = browser.contexts[0].pages[0] if browser.contexts else await browser.new_context().new_page()

    try:
        # 1. Home page로 이동
        await page.goto(f"{CFN_URL}/#home", wait_until="domcontentloaded")

        await page.wait_for_timeout(5000)

        # 2. Cart 확인
        view_cart_button = page.locator("#viewCart")
        await view_cart_button.wait_for(state="visible")
        await view_cart_button.click()
        await page.wait_for_timeout(2000)

    except Exception as error:
        print(f"Error during navigation: {error}")
        raise

세션을 종료합니다.

In [ ]:
stoped_session = browser_cli.stop_browser_session(browserIdentifier=browser_id, sessionId=session_id)
stoped_session

### 3. 세션 녹화 다운로드(선택 사항)

브라우저에 bucket 구성을 추가했으므로 브라우저 탐색 내용을 다운로드해 재현할 수 있는 metadata가 있습니다.

S3 bucket과 key의 file 목록을 확인합니다. S3 key는 session ID입니다.

In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=f"browser_recordings/{session_id}")

for obj in response.get("Contents", []):
    key = obj["Key"]
    if key.endswith(".gz"):
        filename = key.split("/")[-1]  # File 이름만 가져오기
        s3.download_file(BUCKET_NAME, key, filename)
        print(f"Downloaded: {filename}")

그런 다음 압축 file을 열어 재현 가능한 형식으로 변환합니다.

In [ ]:
import gzip

# 압축을 해제하고 event 읽기
events = []
with gzip.open(filename, "rt") as f:
    for line in f:
        line = line.strip()
        if line:  # 빈 줄 건너뛰기
            events.append(json.loads(line))

# rrweb용 JSON으로 저장
with open("events.json", "w") as f:
    json.dump(events, f)

마지막으로 녹화를 재현합니다.

In [ ]:
from IPython.display import HTML
import json

# Event 불러오기
with open("events.json", "r") as f:
    events = json.load(f)

# Inline event를 포함한 HTML 생성
html = f"""
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/rrweb-player@latest/dist/style.css"/>
<div id="player"></div>
<script src="https://cdn.jsdelivr.net/npm/rrweb-player@latest/dist/index.js"></script>
<script>
    new rrwebPlayer({{
        target: document.getElementById('player'),
        props: {{ events: {json.dumps(events)} }}
    }});
</script>
"""

HTML(html)

### 4. 정리(선택 사항)

사용자 지정 AgentCore Browser와 Profile을 삭제합니다.

In [ ]:
browser_boto3.delete_browser(browserId=browser_id)

In [ ]:
browser_boto3.delete_browser_profile(profileId=profile_id)